In [7]:
import pandas as pd
import os

In [8]:
events = pd.read_csv("../data/events.csv")

In [9]:
f = events.groupby("event")["visitorid"].nunique()
funnel_summary = pd.DataFrame({
    "step":  ["Просмотры", "Корзина", "Покупки"],
    "users": [f["view"], f["addtocart"], f["transaction"]],
})
funnel_summary["pct_of_views"] = funnel_summary["users"] / funnel_summary["users"].iloc[0] * 100


In [10]:
events["day"] = pd.to_datetime(events["timestamp"], unit="ms").dt.floor("D")
act = events[["visitorid", "day"]].drop_duplicates()
act["day_n"] = (act["day"] - act.groupby("visitorid")["day"].transform("min")).dt.days
cohort = act["visitorid"].nunique()
retention_curve = (act.groupby("day_n")["visitorid"].nunique() / cohort * 100).loc[0:30].reset_index()
retention_curve.columns = ["day_n", "retention_pct"]


In [14]:
ab = pd.read_csv("../data/cookie_cats.csv")
ab_summary = ab.groupby("version").agg(
    users=("retention_7", "count"),
    ret1_pct=("retention_1", "mean"),
    ret7_pct=("retention_7", "mean"),
).reset_index()
ab_summary[["ret1_pct", "ret7_pct"]] *= 100



In [15]:
os.makedirs("../outputs", exist_ok=True)
funnel_summary.to_csv("../outputs/funnel_summary.csv", index=False)
retention_curve.to_csv("../outputs/retention_curve.csv", index=False)
ab_summary.to_csv("../outputs/ab_summary.csv", index=False)


In [16]:
print("готово, файлы в outputs/")
funnel_summary

готово, файлы в outputs/


,step,users,pct_of_views
0,Просмотры,1404179,100.00000
1,Корзина,37722,2.68641
2,Покупки,11719,0.83458


In [17]:
ab_long = ab_summary.melt(
    id_vars="version",
    value_vars=["ret1_pct", "ret7_pct"],
    var_name="metric",
    value_name="value"
)
ab_long.to_csv("../outputs/ab_summary_long.csv", index=False)
print("длинная таблица сохранена")
ab_long

длинная таблица сохранена


,version,metric,value
0,gate_30,ret1_pct,44.818792
1,gate_40,ret1_pct,44.228275
2,gate_30,ret7_pct,19.020134
3,gate_40,ret7_pct,18.200004
